<a href="https://colab.research.google.com/github/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/blob/main/WHSAT_dashboad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# WHSAT_Comprehensive_Dashboard.ipynb

# Cell 1: Install and import required libraries
!pip install plotly kaleido streamlit pyngrok matplotlib seaborn numpy scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Cell 2: Mount Google Drive and load all saved results
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Load all saved files
save_dir = "/content/drive/MyDrive/WHSAT/"

# Load transformer results
transformer_probs = np.load(f"{save_dir}transformer_probs.npy")
y_test = np.load(f"{save_dir}y_test.npy")
test_indices = np.load(f"{save_dir}test_indices.npy")

# Load Bayesian results
mean_probs = np.load(f"{save_dir}bayesian_mean_probs.npy")
std_probs = np.load(f"{save_dir}bayesian_std_probs.npy")

# Load fusion model and features
import joblib
fusion_model = joblib.load(f"{save_dir}fusion_model.pkl")
kg_features = pd.read_csv(f"{save_dir}kg_features.csv")
kg_test = kg_features.iloc[test_indices]

# Load original data for reference
df = pd.read_csv("https://raw.githubusercontent.com/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/main/cleaned_safety_data_set_A.csv")
data_set = df[['description', 'severity', 'critical_risk']].dropna()
data_set['critical_risk_label'] = data_set['critical_risk'].astype('category').cat.codes
risk_labels = dict(enumerate(data_set['critical_risk'].astype('category').cat.categories))

print("✅ All data loaded successfully!")

# Cell 3: Create predictions and metrics
# Transformer predictions
transformer_preds = np.argmax(transformer_probs, axis=1)

# Bayesian predictions
bayesian_preds = np.argmax(mean_probs, axis=1)
bayesian_uncertainty = std_probs.mean(axis=1)

# Fusion predictions (using pre-trained model)
fusion_features = np.hstack([
    transformer_probs,
    mean_probs,
    std_probs,
    kg_test.values
])
fusion_preds = fusion_model.predict(fusion_features)

# Calculate metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def get_metrics(y_true, y_pred, model_name):
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision (macro)': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall (macro)': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'F1-Score (macro)': f1_score(y_true, y_pred, average='macro', zero_division=0)
    }

metrics = [
    get_metrics(y_test, transformer_preds, 'Transformer (DistilBERT)'),
    get_metrics(y_test, bayesian_preds, 'Bayesian Neural Network'),
    get_metrics(y_test, fusion_preds, 'Fusion Model (All Features)')
]

metrics_df = pd.DataFrame(metrics)
print(metrics_df)

# Cell 4: Create comprehensive dashboard visualizations

# 4.1 Model Performance Comparison
fig1 = make_subplots(rows=1, cols=2,
                     subplot_titles=('Accuracy Comparison', 'F1-Score Comparison'))

models = metrics_df['Model'].tolist()
accuracy = metrics_df['Accuracy'].tolist()
f1_scores = metrics_df['F1-Score (macro)'].tolist()

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig1.add_trace(go.Bar(x=models, y=accuracy, marker_color=colors, text=[f'{x:.3f}' for x in accuracy], textposition='auto'), row=1, col=1)
fig1.add_trace(go.Bar(x=models, y=f1_scores, marker_color=colors, text=[f'{x:.3f}' for x in f1_scores], textposition='auto'), row=1, col=2)

fig1.update_layout(title='Model Performance Comparison', showlegend=False, height=500)
fig1.show()

# 4.2 Confusion Matrices
fig2 = make_subplots(rows=1, cols=3,
                     subplot_titles=('Transformer Confusion Matrix',
                                   'Bayesian Confusion Matrix',
                                   'Fusion Model Confusion Matrix'))

for idx, (preds, title) in enumerate([(transformer_preds, 'Transformer'),
                                        (bayesian_preds, 'Bayesian'),
                                        (fusion_preds, 'Fusion')]):
    cm = confusion_matrix(y_test, preds)
    heatmap = go.Heatmap(z=cm,
                          x=list(range(len(cm))),
                          y=list(range(len(cm))),
                          colorscale='Blues',
                          showscale=True if idx==0 else False)
    fig2.add_trace(heatmap, row=1, col=idx+1)

fig2.update_layout(title='Confusion Matrices Comparison', height=500)
fig2.show()

# 4.3 Bayesian Uncertainty Analysis
fig3 = make_subplots(rows=2, cols=2,
                     subplot_titles=('Uncertainty Distribution',
                                   'Uncertainty by Prediction Correctness',
                                   'Top 5 High Uncertainty Samples',
                                   'Confidence vs Correctness'))

# Uncertainty distribution
fig3.add_trace(go.Histogram(x=bayesian_uncertainty, nbinsx=30, marker_color='steelblue'), row=1, col=1)

# Uncertainty by correctness
correct_mask = (bayesian_preds == y_test)
uncertainty_correct = bayesian_uncertainty[correct_mask]
uncertainty_wrong = bayesian_uncertainty[~correct_mask]

fig3.add_trace(go.Box(y=uncertainty_correct, name='Correct', marker_color='green'), row=1, col=2)
fig3.add_trace(go.Box(y=uncertainty_wrong, name='Wrong', marker_color='red'), row=1, col=2)

# Top high uncertainty samples
high_uncertainty_indices = np.argsort(bayesian_uncertainty)[-5:][::-1]
high_uncertainty_values = bayesian_uncertainty[high_uncertainty_indices]
fig3.add_trace(go.Bar(x=[f'Sample {i}' for i in high_uncertainty_indices],
                       y=high_uncertainty_values, marker_color='coral'), row=2, col=1)

# Confidence distribution for correct vs wrong
max_probs = np.max(mean_probs, axis=1)
conf_correct = max_probs[correct_mask]
conf_wrong = max_probs[~correct_mask]

fig3.add_trace(go.Histogram(x=conf_correct, name='Correct', opacity=0.7, marker_color='green'), row=2, col=2)
fig3.add_trace(go.Histogram(x=conf_wrong, name='Wrong', opacity=0.7, marker_color='red'), row=2, col=2)

fig3.update_layout(title='Bayesian Model Uncertainty Analysis', height=800, showlegend=True)
fig3.update_xaxes(title_text='Uncertainty', row=1, col=1)
fig3.update_xaxes(title_text='Prediction Confidence', row=2, col=2)
fig3.update_yaxes(title_text='Count', row=2, col=2)

fig3.show()

# 4.4 Per-class Performance Analysis
fig4 = make_subplots(rows=1, cols=2,
                     subplot_titles=('Per-Class F1-Score Comparison',
                                   'Class Distribution'))

# Extract per-class metrics
class_labels = list(range(len(np.unique(y_test))))
report_transformer = classification_report(y_test, transformer_preds, output_dict=True, zero_division=0)
report_bayesian = classification_report(y_test, bayesian_preds, output_dict=True, zero_division=0)
report_fusion = classification_report(y_test, fusion_preds, output_dict=True, zero_division=0)

f1_transformer = [report_transformer[str(c)]['f1-score'] for c in class_labels if str(c) in report_transformer]
f1_bayesian = [report_bayesian[str(c)]['f1-score'] for c in class_labels if str(c) in report_bayesian]
f1_fusion = [report_fusion[str(c)]['f1-score'] for c in class_labels if str(c) in report_fusion]

# Only include classes that exist
valid_classes = [c for c in class_labels if str(c) in report_transformer]
valid_class_labels = valid_classes

fig4.add_trace(go.Scatter(x=valid_class_labels, y=f1_transformer, mode='lines+markers',
                           name='Transformer', line=dict(color='blue')), row=1, col=1)
fig4.add_trace(go.Scatter(x=valid_class_labels, y=f1_bayesian, mode='lines+markers',
                           name='Bayesian', line=dict(color='orange')), row=1, col=1)
fig4.add_trace(go.Scatter(x=valid_class_labels, y=f1_fusion, mode='lines+markers',
                           name='Fusion', line=dict(color='green')), row=1, col=1)

# Class distribution
class_counts = pd.Series(y_test).value_counts().sort_index()
fig4.add_trace(go.Bar(x=class_counts.index, y=class_counts.values, marker_color='purple'), row=1, col=2)

fig4.update_layout(title='Per-Class Performance Analysis', height=500)
fig4.update_xaxes(title_text='Severity Level', row=1, col=1)
fig4.update_xaxes(title_text='Severity Level', row=1, col=2)
fig4.update_yaxes(title_text='F1-Score', row=1, col=1)
fig4.update_yaxes(title_text='Count', row=1, col=2)
fig4.show()

# 4.5 Knowledge Graph Insights
fig5 = make_subplots(rows=1, cols=2,
                     specs=[[{'type':'domain'}, {'type':'xy'}]],
                     subplot_titles=('Missing Safety Control Flag Distribution',
                                   'Graph Degree vs Severity'))

# Missing harness flag analysis
missing_harness = kg_test['missing_harness_flag'].values
fig5.add_trace(go.Pie(labels=['Has Harness', 'Missing Harness'],
                       values=[np.sum(missing_harness==0), np.sum(missing_harness==1)],
                       marker_colors=['lightgreen', 'salmon']), row=1, col=1)

# Graph degree vs severity analysis
graph_degree = kg_test['graph_degree'].values
severity_levels = y_test

# Create box plot
unique_severities = np.unique(severity_levels)
box_data = [graph_degree[severity_levels == s] for s in unique_severities]

for s, data in zip(unique_severities, box_data):
    fig5.add_trace(go.Box(y=data, name=f'Severity {s}', boxmean='sd'), row=1, col=2)

fig5.update_layout(title='Knowledge Graph Insights', height=500)
fig5.update_xaxes(title_text='Severity Level', row=1, col=2)
fig5.update_yaxes(title_text='Graph Degree Centrality', row=1, col=2)
fig5.show()

# 4.6 Feature Importance from Fusion Model
if hasattr(fusion_model, 'feature_importances_'):
    fig6 = go.Figure()

    feature_importance = fusion_model.feature_importances_
    feature_categories = ['Transformer Probs'] * transformer_probs.shape[1] + \
                        ['Bayesian Mean'] * mean_probs.shape[1] + \
                        ['Bayesian Std'] * std_probs.shape[1] + \
                        ['KG Features'] * kg_test.shape[1]

    # Aggregate importance by category
    importance_by_category = {}
    for imp, cat in zip(feature_importance, feature_categories):
        if cat not in importance_by_category:
            importance_by_category[cat] = []
        importance_by_category[cat].append(imp)

    avg_importance = {cat: np.mean(imps) for cat, imps in importance_by_category.items()}

    fig6.add_trace(go.Bar(x=list(avg_importance.keys()),
                          y=list(avg_importance.values()),
                          marker_color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'],
                          text=[f'{v:.3f}' for v in avg_importance.values()],
                          textposition='auto'))

    fig6.update_layout(title='Feature Category Importance (Random Forest)',
                       xaxis_title='Feature Category',
                       yaxis_title='Mean Importance Score',
                       height=500)
    fig6.show()

# Cell 5: Create summary statistics table
summary_stats = pd.DataFrame({
    'Metric': [
        'Total Test Samples',
        'Number of Severity Classes',
        'Transformer Accuracy',
        'Bayesian Accuracy',
        'Fusion Model Accuracy',
        'Average Bayesian Uncertainty',
        'Fusion Model F1-Score',
        'Missing Harness Count',
        'Graph Density'
    ],
    'Value': [
        len(y_test),
        len(np.unique(y_test)),
        f"{accuracy_score(y_test, transformer_preds):.3f}",
        f"{accuracy_score(y_test, bayesian_preds):.3f}",
        f"{accuracy_score(y_test, fusion_preds):.3f}",
        f"{bayesian_uncertainty.mean():.4f}",
        f"{f1_score(y_test, fusion_preds, average='macro'):.3f}",
        int(missing_harness.sum()),
        f"{'N/A'}" # Removed nx.density(G) because G is not defined in this scope
    ]
})

print("\n" + "="*50)
print("SUMMARY STATISTICS")
print("="*50)
print(summary_stats.to_string(index=False))

# Cell 6: Create interactive HTML dashboard
from IPython.display import HTML

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>WHSAT - Workplace Safety Analytics Dashboard</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background-color: #f5f5f5;
        }}
        .container {{
            max-width: 1200px;
            margin: 0 auto;
            background-color: white;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }}
        h1 {{
            color: #2c3e50;
            text-align: center;
            border-bottom: 3px solid #3498db;
            padding-bottom: 10px;
        }}
        h2 {{
            color: #34495e;
            margin-top: 30px;
        }}
        .metric-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 20px;
            margin: 20px 0;
        }}
        .metric-card {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 20px;
            border-radius: 10px;
            text-align: center;
        }}
        .metric-value {{
            font-size: 2em;
            font-weight: bold;
        }}
        .metric-label {{
            font-size: 0.9em;
            margin-top: 10px;
        }}
        .best-model {{
            background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%);
        }}
        .summary-table {{
            width: 100%;
            border-collapse: collapse;
            margin: 20px 0;
        }}
        .summary-table th, .summary-table td {{
            border: 1px solid #ddd;
            padding: 12px;
            text-align: left;
        }}
        .summary-table th {{
            background-color: #3498db;
            color: white;
        }}
        .summary-table tr:nth-child(even) {{
            background-color: #f2f2f2;
        }}
        .insight {{
            background-color: #ecf0f1;
            padding: 15px;
            border-left: 4px solid #3498db;
            margin: 20px 0;
            border-radius: 5px;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>🏭 Workplace Safety Analytics Dashboard (WHSAT)</h1>
        <p style="text-align: center; color: #7f8c8d;">Multi-Model Fusion for Industrial Safety Risk Assessment</p>

        <div class="metric-grid">
            <div class="metric-card">
                <div class="metric-value">{len(y_test)}</div>
                <div class="metric-label">Test Samples</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">{len(np.unique(y_test))}</div>
                <div class="metric-label">Severity Levels</div>
            </div>
            <div class="metric-card best-model">
                <div class="metric-value">{accuracy_score(y_test, fusion_preds):.1%}</div>
                <div class="metric-label">Best Model Accuracy</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">{f1_score(y_test, fusion_preds, average='macro'):.3f}</div>
                <div class="metric-label">Fusion Model F1-Score</div>
            </div>
        </div>

        <h2>📊 Key Insights</h2>
        <div class="insight">
            <strong>🎯 Model Performance:</strong> The fusion model achieves {accuracy_score(y_test, fusion_preds):.1%} accuracy,
            significantly outperforming individual models, demonstrating the value of combining text, uncertainty,
            and knowledge graph features.
        </div>

        <div class="insight">
            <strong>⚠️ Risk Analysis:</strong> {int(missing_harness.sum())} incidents identified as missing critical safety controls
            (harness for fall protection), highlighting areas for safety intervention.
        </div>

        <div class="insight">
            <strong>📈 Uncertainty Insights:</strong> Average prediction uncertainty for the Bayesian model is {bayesian_uncertainty.mean():.4f},
            with high-uncertainty samples showing {len(uncertainty_wrong)} misclassifications.
        </div>

        <h2>📋 Model Comparison Summary</h2>
        {metrics_df.round(4).to_html(index=False, classes='summary-table')}

        <h2>🎯 Risk Category Mapping</h2>
        <div class="insight">
            <strong>Critical Risk Categories:</strong><br>
            {', '.join(list(risk_labels.values())[:10])}...
        </div>

        <h2>💡 Recommendations</h2>
        <ul>
            <li><strong>Safety Interventions:</strong> Focus on high-risk categories with lower F1-scores</li>
            <li><strong>Uncertainty Monitoring:</strong> Flag high-uncertainty predictions for human review</li>
            <li><strong>Knowledge Graph Enhancement:</strong> Expand hazard-control relationships for better risk prediction</li>
            <li><strong>Model Deployment:</strong> Use fusion model for real-time safety alerts</li>
        </ul>

        <hr>
        <p style="text-align: center; color: #95a5a6; font-size: 0.8em;">
            Dashboard generated from WHSAT analysis pipeline | Multi-Model Fusion Framework
        </p>
    </div>
</body>
</html>
"""

# Display HTML dashboard
from IPython.display import display, HTML
display(HTML(html_content))

# Cell 7: Save dashboard as HTML file
with open('/content/whsat_dashboard.html', 'w') as f:
    f.write(html_content)

from google.colab import files
files.download('/content/whsat_dashboard.html')
print("✅ Dashboard saved and downloaded as 'whsat_dashboard.html'")

# Cell 8: Print final evaluation console output
print("\n" + "="*70)
print("FINAL MODEL EVALUATION RESULTS")
print("="*70)
print("\n📊 Detailed Classification Report - Fusion Model:")
print(classification_report(y_test, fusion_preds, target_names=[str(risk_labels.get(i, f'Class_{i}')) for i in range(len(np.unique(y_test)))]))

print("\n" + "="*70)
print("TOP HIGH UNCERTAINTY SAMPLES FOR REVIEW")
print("="*70)
high_uncertainty_indices = np.argsort(bayesian_uncertainty)[-10:][::-1]
for i, idx in enumerate(high_uncertainty_indices[:5], 1):
    print(f"\n{i}. Sample Index: {idx}")
    print(f"   True Severity: {y_test[idx]}")
    print(f"   Predicted: {bayesian_preds[idx]}")
    print(f"   Uncertainty: {bayesian_uncertainty[idx]:.4f}")
    if idx < len(data_set):
        desc_preview = data_set.iloc[test_indices[idx]]['description'][:150]
        print(f"   Description: {desc_preview}...")

print("\n✅ Dashboard analysis complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 28.7 MB/s eta 0:00:00
Mounted at /content/drive
✅ All data loaded successfully!
                         Model  Accuracy  Precision (macro)  Recall (macro)  \
0     Transformer (DistilBERT)       0.4           0.021053        0.052632   
1      Bayesian Neural Network       0.0           0.000000        0.000000   
2  Fusion Model (All Features)       1.0           1.000000        1.000000   

   F1-Score (macro)  
0          0.030075  
1          0.000000  
2          1.000000  



SUMMARY STATISTICS
                      Metric  Value
          Total Test Samples     85
  Number of Severity Classes     19
        Transformer Accuracy  0.400
           Bayesian Accuracy  0.000
       Fusion Model Accuracy  1.000
Average Bayesian Uncertainty 0.0669
       Fusion Model F1-Score  1.000
       Missing Harness Count      3
               Graph Density    N/A


Model,Accuracy,Precision (macro),Recall (macro),F1-Score (macro)
Transformer (DistilBERT),0.4,0.0211,0.0526,0.0301
Bayesian Neural Network,0.0,0.0000,0.0000,0.0000
Fusion Model (All Features),1.0,1.0000,1.0000,1.0000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Dashboard saved and downloaded as 'whsat_dashboard.html'

FINAL MODEL EVALUATION RESULTS

📊 Detailed Classification Report - Fusion Model:
                                    precision    recall  f1-score   support

                   
Not applicable       1.00      1.00      1.00         2
                              Bees       1.00      1.00      1.00         7
Blocking and isolation of energies       1.00      1.00      1.00         7
                              Burn       1.00      1.00      1.00         1
               Chemical substances       1.00      1.00      1.00         3
                    Confined space       1.00      1.00      1.00         3
                               Cut       1.00      1.00      1.00         1
                  Electrical Shock       1.00      1.00      1.00         1
           Electrical installation       1.00      1.00      1.00         5
                              Fall       1.00      1.00      1.00        34
                   Fal